In [2]:
import numpy as np
import torch
import torchvision.transforms as T
from decord import VideoReader, cpu
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer

In [3]:
# transforming images or video frames into normalized tensors
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

In [4]:
def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

In [5]:

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    # calculate the existing image aspect ratio
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    # find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    # calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    # resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        # split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

In [6]:
path = './pretrained/InternVL2_5-1B'
model = AutoModel.from_pretrained(
    path,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_flash_attn=True,
    trust_remote_code=True).eval().cuda()
tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True, use_fast=False, attn_implementations="flash_attention2")

/home/znyd/hacking/edu-cut/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Sliding Window Attention is enabled but not implemented for `eager`; unexpected results may be encountered.


FlashAttention2 is not installed.


In [ ]:
# def load_image(image_file, input_size=448, max_num=12):
#     image = Image.open(image_file).convert('RGB')
#     transform = build_transform(input_size=input_size)
#     images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
#     pixel_values = [transform(image) for image in images]
#     pixel_values = torch.stack(pixel_values)
#     return pixel_values

In [ ]:
# set the max number of tiles in `max_num`
# pixel_values = load_image('./media/online_class.png', max_num=12).to(torch.bfloat16).cuda()
# generation_config = dict(max_new_tokens=1024, do_sample=False)

In [ ]:
# single-image single-round conversation (单图单轮对话)
# question = '<image>\nPlease describe the educational content on the image shortly in well formated way.'
# response = model.chat(tokenizer, pixel_values, question, generation_config)
# print(f'User: {question}\nAssistant: {response}')

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


User: <image>
Please describe the educational content on the image shortly in well formated way.
Assistant: The image shows a mathematical explanation of a logical address calculation. The address is given as \((A4FB : 4872)\) and is broken down into segments:

1. Physical Address: \(A4FB0h + 4872 = (A982h2h)\)
2. First P.A. segment: \(A4FB0h\)
3. Second P.A. segment: \(A4FB0 + 1 = A4FB1h\)
4. Third P.A. segment: \(A4FD0 + 2 = A4FD2h\)

The explanation uses binary numbers to represent hexadecimal values, and the address is calculated by adding the hexadecimal values of the first and second segments, then adding 1 to the first segment and 2 to the second segment.


In [14]:
# video multi-round conversation (视频多轮对话)
def get_index(bound, fps, max_frame, first_idx=0, num_segments=32):
    if bound:
        start, end = bound[0], bound[1]
    else:
        start, end = -100000, 100000
    start_idx = max(first_idx, round(start * fps))
    end_idx = min(round(end * fps), max_frame)
    seg_size = float(end_idx - start_idx) / num_segments
    frame_indices = np.array([
        int(start_idx + (seg_size / 2) + np.round(seg_size * idx))
        for idx in range(num_segments)
    ])
    return frame_indices

def load_video(video_path, bound=None, input_size=448, max_num=1, num_segments=32):
    vr = VideoReader(video_path, ctx=cpu(0), num_threads=1)
    max_frame = len(vr) - 1
    fps = float(vr.get_avg_fps())

    pixel_values_list, num_patches_list = [], []
    transform = build_transform(input_size=input_size)
    frame_indices = get_index(bound, fps, max_frame, first_idx=0, num_segments=num_segments)
    for frame_index in frame_indices:
        img = Image.fromarray(vr[frame_index].asnumpy()).convert('RGB')
        img = dynamic_preprocess(img, image_size=input_size, use_thumbnail=True, max_num=max_num)
        pixel_values = [transform(tile) for tile in img]
        pixel_values = torch.stack(pixel_values)
        num_patches_list.append(pixel_values.shape[0])
        pixel_values_list.append(pixel_values)
    pixel_values = torch.cat(pixel_values_list)
    return pixel_values, num_patches_list


In [22]:
generation_config = dict(max_new_tokens=1024, do_sample=False)
video_path = './media/ronaldo_goal.mp4'
pixel_values, num_patches_list = load_video(video_path, num_segments=20, max_num=1)
pixel_values = pixel_values.to(torch.bfloat16).cuda()
video_prefix = ''.join([f'Frame{i+1}: <image>\n' for i in range(len(num_patches_list))])

prompt = """Please describe this video clip in complete detail. Include all visible activities, actions, people, and objects. Explain what is happening 
         over time, what roles different people or objects play, and how the scene changes. Describe the environment (indoor or outdoor, location type, 
         lighting, background elements), any emotions or expressions, and interactions between subjects. Mention any relevant objects, movement, gestures, or 
         background events that may be important. Your goal is to provide a full understanding of everything shown in this short video segment."""
prompt_action_focused ="""
You are given a video clip of 8 seconds extracted from a longer 25-second video. For this clip, please analyze every second in detail. For each one-second interval, provide a bullet-point list covering:
• All observable activities and actions (e.g., running, jumping, backflipping, sitting, interacting)
• Specific details of any rapid or acrobatic movements (if a person is performing an action like a back flip, mention exactly when it happens and describe the movement and body posture)
• A description of the environment and background (location, lighting, objects, and scene changes)
• Any additional noteworthy details (such as gestures, facial expressions, or subtle movements)
Ensure that you examine every moment carefully and include every detail—even if it appears fleeting. Then, compile your segmented bullet lists into one comprehensive description for the entire 8-second clip. Your output will be used for indexing so that each 5-second segment can have its own semantic embedding.
"""

# question = video_prefix + 'Describe only activities on this video like jumping, back flip, dancing etc Don\'t describe frames overall description, only activities  Don\'t repeat.'
question = video_prefix +prompt 
# Frame1: <image>\nFrame2: <image>\n...\nFrame8: <image>\n{question}
response, history = model.chat(tokenizer, pixel_values, question+" Don\'t repeat", generation_config,
                               num_patches_list=num_patches_list, history=None, return_history=True)
print(f'User: {question}\nAssistant: {response}')

#print(history)
# question = 'Describe this video in detail. Don\'t repeat.'
# response, history = model.chat(tokenizer, pixel_values, question, generation_config,
#                                num_patches_list=num_patches_list, history=history, return_history=True)
# print(f'User: {question}\nAssistant: {response}')

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


User: Frame1: <image>
Frame2: <image>
Frame3: <image>
Frame4: <image>
Frame5: <image>
Frame6: <image>
Frame7: <image>
Frame8: <image>
Frame9: <image>
Frame10: <image>
Frame11: <image>
Frame12: <image>
Frame13: <image>
Frame14: <image>
Frame15: <image>
Frame16: <image>
Frame17: <image>
Frame18: <image>
Frame19: <image>
Frame20: <image>
Please describe this video clip in complete detail. Include all visible activities, actions, people, and objects. Explain what is happening 
         over time, what roles different people or objects play, and how the scene changes. Describe the environment (indoor or outdoor, location type, 
         lighting, background elements), any emotions or expressions, and interactions between subjects. Mention any relevant objects, movement, gestures, or 
         background events that may be important. Your goal is to provide a full understanding of everything shown in this short video segment.
Assistant: The video captures a soccer match between Arsenal and M

In [19]:
#ssingle-image single-round conversation (单图单轮对话)
question = 'give me a prompt for InternVL2.5 so it can find every details, activity from a video clip, also tell me how long video clip it can handle?'
response = model.chat(tokenizer, question, generation_config)
print(f'User: {question}\nAssistant: {response}')

TypeError: InternVLChatModel.chat() missing 1 required positional argument: 'generation_config'